# 05 — Full JWT passthrough

The agent forwards the user's full Keycloak-issued JWT to the tool, in the
`Authorization: Bearer ...` header. The tool validates the JWT signature
against Keycloak's JWKS, extracts the claims, and uses them to scope the
response.

Now the tool has *cryptographic proof* of who the user is. A compromised
agent that wants to call the tool as carlo would need carlo's JWT, which
means it would need to either steal it or trick Keycloak into issuing one
— a much higher bar than spoofing an `X-User-Id` header.

## The gap from pattern 4

Pattern 4 fixed the *information* problem (the tool now knows who's
asking) but not the *trust* problem (the tool just believes whatever the
agent says). Pattern 5 fixes the trust problem by having the tool
verify the JWT signature against Keycloak's published keys.

The JWT is issued by Keycloak when the user authenticates. It contains
the user's identity (`sub`, `preferred_username`), their custom claims
(`role`, `department`, `reports_to`), and the audience(s) it's valid for
(`aud`). It's signed with Keycloak's private key, so any tampering is
detectable.

In [ ]:
from shared.agent import Agent
from shared.auth import JWTPassthroughAuth, fetch_user_jwt
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw, show_token
from shared.config import EXPENSE_SERVICE_URL, DOCUMENT_SERVICE_URL

strategy = JWTPassthroughAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## The token the agent will forward

Look at the JWT before it goes to the service. Note in particular the
`aud` field — this is what the gap at the end of this notebook will be
about.

In [ ]:
alice_jwt = fetch_user_jwt("alice")
show_token(alice_jwt, label="alice user JWT (what JWTPassthroughAuth forwards as-is)")

## Same prompts, three users — now with cryptographically proven identity

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

In [ ]:
result_carlo = run_as("carlo", "Show me all the expenses across the company.", agent)
show_what_tool_saw(EXPENSE_SERVICE_URL)

## The problem: this token is too broad

Look at the `aud` field on the token above:

    aud = ['expense-service-client', 'document-service-client']

That's saying "this token is valid for both the expense-service AND the
document-service". The agent might be calling expense-service right now,
but if expense-service ever logs the token, gets compromised, or makes an
outbound request that somehow exposes it — the same token works against
document-service too.

The general principle: **a credential should be valid for the smallest
scope that still lets it do its job.** A token meant for expense-service
should not be usable against document-service. A token meant to read
should not be usable to write.

Let's prove the breadth concretely by using the same alice token to call
the document-service:

In [ ]:
import httpx
r = httpx.get(
    f"{DOCUMENT_SERVICE_URL}/documents",
    headers={"Authorization": f"Bearer {alice_jwt}"},
    timeout=5.0,
)
body = r.json()
print(f"identity_method: {body['identity_method']}")
print(f"detail:          {body['identity_detail']}")
print(f"document count:  {body['count']}")
print()
print("Same JWT, different service. Both accept it. This is the breadth")
print("problem that pattern 6 fixes via token exchange.")

## Tradeoffs

- **Where authz lives:** in the tool, based on validated JWT claims. The
  agent is now just a transport layer for the user's identity.
- **Tool sees real user:** yes, with cryptographic proof. Method is `jwt`.
- **Cryptographic proof of identity:** **yes**. The signature is verified
  against Keycloak's JWKS (cached at startup). A forged token would fail
  validation.
- **Least privilege:** **no** — the token is valid for *every* service
  the agent might call. Stealing it from one service compromises all of
  them.
- **Audit trail:** the service can log `sub`, `preferred_username`, and
  any other claims. "alice viewed expenses at 2026-04-08 16:23 UTC" is
  a real, attributable fact.
- **The blast radius problem:** if any one service that handles this token
  is compromised, the attacker has a working token for every other service
  the same token covers. Token theft is a near-universal failure mode of
  bearer-token systems; reducing the value of a stolen token by narrowing
  its scope is the single highest-leverage mitigation.

## What's still missing?

We have proven identity, but the credential is too broad. The fix is to
narrow the audience: instead of forwarding the user's broad JWT, exchange
it for a new token that's scoped to *only* the service we're about to
call.

That's RFC 8693 Token Exchange, and Keycloak has supported it since 26.2
as a GA feature called Standard Token Exchange v2. The next notebook
(`06_token_exchange`) is the gold-standard pattern: identity preserved,
audience narrowed, agent identified as the requester via `azp`.